### Remove duplicates

In [15]:
from pathlib import Path
import shutil
import hashlib
from PIL import Image

from pathlib import Path
import numpy
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np


In [3]:
RAW_DIR = Path("../dataset/raw/CoconutLeaves")
CLEAN_DIR = Path("../dataset/cleaned/CoconutLeaves")
SPLIT_DIR = Path("../dataset/split")

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
seen_hashes = set()

copied_images = 0
duplicate_images = 0
invalid_images = 0

valid_extensions = {".jpg", ".jpeg", ".png"}

for class_dir in sorted(RAW_DIR.iterdir()):

    if not class_dir.is_dir():
        continue

    destination_class = CLEAN_DIR / class_dir.name
    destination_class.mkdir(parents=True, exist_ok=True)

    for image_path in class_dir.iterdir():

        if not image_path.is_file():
            continue

        if image_path.suffix.lower() not in valid_extensions:
            continue

        try:
            with Image.open(image_path) as image:
                image.load()

            with open(image_path, "rb") as file:
                image_hash = hashlib.md5(file.read()).hexdigest()

            if image_hash in seen_hashes:
                duplicate_images += 1
                continue

            seen_hashes.add(image_hash)

            shutil.copy2(
                image_path,
                destination_class / image_path.name
            )

            copied_images += 1

        except Exception:
            invalid_images += 1

In [5]:
print("Unique images copied:", copied_images)
print("Duplicates removed:", duplicate_images)
print("Invalid images skipped:", invalid_images)

Unique images copied: 8912
Duplicates removed: 35
Invalid images skipped: 0


In [6]:
for class_dir in sorted(CLEAN_DIR.iterdir()):

    if class_dir.is_dir():

        count = len([
            file
            for file in class_dir.iterdir()
            if file.suffix.lower() in valid_extensions
        ])

        print(f"{class_dir.name}: {count}")

CCI_Caterpillars: 990
CCI_Leaflets: 795
Gray Leaf Spot: 2132
Healthy_Leaves: 123
Leaf Rot: 1642
WCLWD_DryingofLeaflets: 1078
WCLWD_Flaccidity: 1068
WCLWD_Yellowing: 1084


### Create the stratified split

In [7]:
TRAIN_DIR = SPLIT_DIR / "train"
VAL_DIR = SPLIT_DIR / "val"
TEST_DIR = SPLIT_DIR / "test"

In [8]:
records = []

for class_dir in sorted(CLEAN_DIR.iterdir()):

    if not class_dir.is_dir():
        continue

    for image_path in class_dir.iterdir():

        if (
            image_path.is_file()
            and image_path.suffix.lower() in valid_extensions
        ):
            records.append({
                "file_path": str(image_path),
                "class": class_dir.name
            })

clean_df = pd.DataFrame(records)

print("Total cleaned images:", len(clean_df))
print("Number of classes:", clean_df["class"].nunique())

clean_df.head()

Total cleaned images: 8912
Number of classes: 8


,file_path,class
0,..\dataset\cleaned\CoconutLeaves\CCI_Caterpill...,CCI_Caterpillars
1,..\dataset\cleaned\CoconutLeaves\CCI_Caterpill...,CCI_Caterpillars
2,..\dataset\cleaned\CoconutLeaves\CCI_Caterpill...,CCI_Caterpillars
3,..\dataset\cleaned\CoconutLeaves\CCI_Caterpill...,CCI_Caterpillars
4,..\dataset\cleaned\CoconutLeaves\CCI_Caterpill...,CCI_Caterpillars


### Split 70% / 30%

In [9]:
train_df, temp_df = train_test_split(
    clean_df,
    test_size=0.30,
    stratify=clean_df["class"],
    random_state=42
)

### Split remaining 30% into 15% + 15%

In [10]:
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["class"],
    random_state=42
)

### Verify the split

In [11]:
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Testing images:", len(test_df))

print(
    "\nTotal:",
    len(train_df) + len(val_df) + len(test_df)
)

Training images: 6238
Validation images: 1337
Testing images: 1337

Total: 8912


### Copy images into the split folders

In [17]:
if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)

TRAIN_DIR.mkdir(parents=True)
VAL_DIR.mkdir(parents=True)
TEST_DIR.mkdir(parents=True)

In [18]:
def copy_split(split_df, destination_dir):

    for _, row in split_df.iterrows():

        source = Path(row["file_path"])

        class_dir = destination_dir / row["class"]
        class_dir.mkdir(parents=True, exist_ok=True)

        shutil.copy2(
            source,
            class_dir / source.name
        )

In [19]:
copy_split(train_df, TRAIN_DIR)
copy_split(val_df, VAL_DIR)
copy_split(test_df, TEST_DIR)

print("Dataset split completed.")

Dataset split completed.


### verify the per-class distribution and calculate class weights

In [20]:
split_summary = pd.DataFrame({
    "Total": clean_df["class"].value_counts(),
    "Train": train_df["class"].value_counts(),
    "Validation": val_df["class"].value_counts(),
    "Test": test_df["class"].value_counts(),
})

split_summary = (
    split_summary
    .sort_index()
    .rename_axis("Class")
    .reset_index()
)

split_summary

,Class,Total,Train,Validation,Test
0,CCI_Caterpillars,990,693,149,148
1,CCI_Leaflets,795,556,120,119
2,Gray Leaf Spot,2132,1492,320,320
3,Healthy_Leaves,123,86,18,19
4,Leaf Rot,1642,1149,246,247
5,WCLWD_DryingofLeaflets,1078,755,161,162
6,WCLWD_Flaccidity,1068,748,160,160
7,WCLWD_Yellowing,1084,759,163,162


In [21]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array(sorted(train_df["class"].unique()))

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["class"]
)

class_weights = {
    i: weight
    for i, weight in enumerate(weights)
}

class_weight_table = pd.DataFrame({
    "class": classes,
    "weight": weights
})

class_weight_table

,class,weight
0,CCI_Caterpillars,1.125180
1,CCI_Leaflets,1.402428
2,Gray Leaf Spot,0.522621
3,Healthy_Leaves,9.066860
4,Leaf Rot,0.678634
5,WCLWD_DryingofLeaflets,1.032781
6,WCLWD_Flaccidity,1.042447
7,WCLWD_Yellowing,1.027339
